# Unit 5：完整演练 —— 端到端 ICA 伪迹去除

## 目标
- 用 mne-sample-data 完成完整 ICA 流程
- 对比 ICA 前后：时域、空间、频域三维评估
- 量化去伪迹效果


### 5.1 加载 + 滤波 + 坏段剔除（一步到位）


In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from mne.preprocessing import ICA

# ── 加载 ──
sample_data_dir = mne.datasets.sample.data_path()
raw_fname = sample_data_dir / 'MEG' / 'sample' / 'sample_audvis_raw.fif'
raw = mne.io.read_raw_fif(raw_fname, preload=True)
raw.pick(['eeg'])
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage)

print(f"原始: {len(raw.ch_names)} 通道, {raw.times[-1]:.0f}s, {raw.info['sfreq']} Hz")

# ── 滤波 ──
raw_filt = raw.copy().filter(l_freq=1.0, h_freq=40.0, fir_design='firwin')
raw_filt = raw_filt.notch_filter(freqs=50, fir_design='firwin')
print("✅ 滤波: 1-40 Hz 带通 + 50 Hz 陷波")

# ── 剔除大伪迹段 ──
reject_criteria = dict(eeg=200e-6)
events = mne.make_fixed_length_events(raw_filt, duration=1.0)
epochs = mne.Epochs(raw_filt, events, tmin=0, tmax=0.99,
                     baseline=None, reject=reject_criteria, preload=True)
print(f"✅ 坏段剔除: {len(events)-len(epochs)}/{len(events)} 段被排除")

# 用 epoch 重建 raw（MNE 内部会自动跳过坏段）
# 这里我们用 epoch 平均来做 ICA 拟合（epoch 已经自动 reject 了坏段）
# 或者直接用 raw 但后面 ICA fit 时传 reject
print(f"保留: {len(epochs)} 段 → 用于 ICA 拟合")


### 5.2 ICA 拟合


In [ ]:
n_components = 20
ica = ICA(
    n_components=n_components,
    method='fastica',
    random_state=42,
    max_iter='auto',
    fit_params=dict(extended=True)
)

print(f"拟合 ICA（{n_components} 成分）...")
# 在 epochs 上拟合（自动跳过坏段）
ica.fit(epochs)
print(f"✅ 完成！迭代: {ica.n_iter_}")
print(f"解释方差比 (ICA 成分): {np.sum(ica.pca_explained_variance_[:n_components]) / np.sum(ica.pca_explained_variance_):.1%}")


### 5.3 成分识别 —— 三视角排查


In [ ]:
# ① 地形图总览
ica.plot_components(picks=range(n_components))
print("→ 找出前额高度集中的成分（眨眼/眼动候选）")


In [ ]:
# ② 前 10 个成分的时间序列
ica.plot_sources(raw_filt, picks=range(min(10, n_components)))
print("→ 找间歇脉冲（眨眼）、规律尖峰（心跳）、持续噪声（肌电）")


In [ ]:
# ③ 逐个查验可疑成分
# 改成你想要细看的成分编号
suspects = [0, 1, 2]  # ← 根据前面的观察修改这个列表
for i in suspects:
    print(f"\n{'='*40}\n成分 {i}:")
    ica.plot_properties(raw_filt, picks=[i])


### 5.4 标记 + 剔除伪迹成分


In [ ]:
# ── 自动检测 ──
eog_indices, eog_scores = ica.find_bads_eog(
    raw_filt, ch_name='EEG 001', threshold=2.5
)
print(f"自动检测眼动成分: {eog_indices}")

# ── 手动标记 ──
# 观察地形图 + 时间序列 + 功率谱后，把你确认的伪迹成分加进去
ica.exclude = list(eog_indices)  # 从自动检测开始

# 示例：如果你看到成分 3 也是眨眼（前额地形图 + 间歇脉冲），加上它
# ica.exclude.append(3)

print(f"\n最终剔除: {ica.exclude}")
print(f"保留成分: {[i for i in range(n_components) if i not in ica.exclude]}")
print(f"剔除比例: {len(ica.exclude)}/{n_components} = {len(ica.exclude)/n_components:.0%}")


In [ ]:
# ── 应用到数据 ──
raw_clean = ica.apply(raw_filt.copy())
print("✅ ICA 伪迹剔除完成！")


### 5.5 三维对比：ICA 前 vs ICA 后


#### ① 时域对比


In [ ]:
# 前额通道 Fp1 的时间序列
ch_idx = raw_filt.ch_names.index('EEG 001')
times = raw_filt.times[:15000]
before = raw_filt.get_data()[ch_idx, :15000] * 1e6
after = raw_clean.get_data()[ch_idx, :15000] * 1e6

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True, sharey=True)
axes[0].plot(times, before, linewidth=0.5, color='crimson', alpha=0.8)
axes[0].set_ylabel('μV')
axes[0].set_title('ICA 前 — EEG 001 (≈Fp1)，注意眨眼大脉冲')
axes[1].plot(times, after, linewidth=0.5, color='forestgreen')
axes[1].set_ylabel('μV')
axes[1].set_title('ICA 后 — EEG 001 (≈Fp1)，眨眼应被抑制')
axes[1].set_xlabel('时间 (s)')
plt.tight_layout(); plt.show()


#### ② 空间对比（各通道方差）


In [ ]:
# 所有通道的方差：ICA 前 vs 后
var_before = raw_filt.get_data().var(axis=1)
var_after = raw_clean.get_data().var(axis=1)

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(raw_filt.ch_names))
width = 0.35
ax.bar(x - width/2, var_before * 1e12, width, label='ICA 前',
       color='crimson', alpha=0.6)
ax.bar(x + width/2, var_after * 1e12, width, label='ICA 后',
       color='forestgreen', alpha=0.6)
ax.set_xticks(x[::3])
ax.set_xticklabels([raw_filt.ch_names[i] for i in range(0, len(x), 3)],
                   rotation=45, fontsize=8)
ax.set_ylabel('方差 (pV²)')
ax.set_title('各通道方差：红=ICA前  绿=ICA后')
ax.legend()
plt.tight_layout(); plt.show()
print("→ 前几个通道（Fp1/Fp2 区域）方差应显著下降")


#### ③ 频域对比（功率谱）


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 前额
raw_filt.plot_psd(picks='EEG 001', fmin=1, fmax=40, ax=axes[0],
                  color='crimson', alpha=0.7, spatial_colors=False)
raw_clean.plot_psd(picks='EEG 001', fmin=1, fmax=40, ax=axes[0],
                   color='forestgreen', alpha=0.7, spatial_colors=False)
axes[0].set_title('EEG 001 (≈Fp1) 功率谱')
axes[0].legend(['ICA 前', 'ICA 后'])

# 枕部（对照）
raw_filt.plot_psd(picks='EEG 060', fmin=1, fmax=40, ax=axes[1],
                  color='crimson', alpha=0.7, spatial_colors=False)
raw_clean.plot_psd(picks='EEG 060', fmin=1, fmax=40, ax=axes[1],
                   color='forestgreen', alpha=0.7, spatial_colors=False)
axes[1].set_title('EEG 060 (≈Oz) 功率谱（对照）')
axes[1].legend(['ICA 前', 'ICA 后'])
plt.tight_layout(); plt.show()
print("→ 理想：Fp1 低频功率大幅下降，Oz 变化很小")


### 5.6 量化评估


In [ ]:
# 计算关键通道的方差变化
fp1_idx = raw_filt.ch_names.index('EEG 001')
fp2_idx = raw_filt.ch_names.index('EEG 002')
oz_idx  = raw_filt.ch_names.index('EEG 060')

vars_before = [raw_filt.get_data()[i].var() for i in [fp1_idx, fp2_idx, oz_idx]]
vars_after  = [raw_clean.get_data()[i].var() for i in [fp1_idx, fp2_idx, oz_idx]]

print("=" * 55)
print(f"{'通道':<12} {'ICA前(pV²)':>12} {'ICA后(pV²)':>12} {'变化':>10}")
print("-" * 55)
for name, vb, va in zip(['EEG001/Fp1', 'EEG002/Fp2', 'EEG060/Oz'],
                          vars_before, vars_after):
    change = (1 - va/vb) * 100
    print(f"{name:<12} {vb*1e12:>10.1f}   {va*1e12:>10.1f}   {change:>+8.1f}%")
print("=" * 55)
print("\n✅ 好的结果：Fp1/Fp2 方差 ↓↓（伪迹去除），Oz 基本不变（脑信号保留）")


### 5.7 保存结果


In [ ]:
ica.save('ica_solution.fif')
print("✅ ICA 方案 → ica_solution.fif")

raw_clean.save('eeg_clean_raw.fif', overwrite=True)
print("✅ 干净 EEG → eeg_clean_raw.fif")

# 后续加载：
# ica = mne.preprocessing.read_ica('ica_solution.fif')
# raw = mne.io.read_raw_fif('eeg_clean_raw.fif')


### 5.8 最终总结

```text
               原始 EEG (含伪迹)
                      │
                      ▼
             ① 带通滤波 1-40 Hz
                      │
                      ▼
             ② 陷波 50 Hz
                      │
                      ▼
             ③ 剔除大伪迹段 (reject >200μV)
                      │
                      ▼
             ④ ICA 拟合 (FastICA, 20 成分)
                      │
                      ▼
             ⑤ 三视角识别 (地形图 + 时序 + PSD)
                      │
                      ▼
             ⑥ 剔除伪迹成分
                      │
                      ▼
               干净 EEG ✨
```

### 🎯 你学会了什么

- [x] ICA 数学原理（鸡尾酒会 → FastICA）
- [x] EEG 常见伪迹的时-频-空三维特征
- [x] MNE ICA 完整流水线（滤波 → reject → fit → 识别 → 剔除）
- [x] 成分可视化：地形图 + 时间序列 + 功率谱
- [x] 端到端实战 + 量化评估

### 🚀 下一步

- 试试 `method='picard'`（比 FastICA 更快更稳定）
- 对比 ICA 和 SSP（信号空间投影）
- 在 ERP/时频分析前做 ICA
- 拿你自己的 EEG 数据练手

### 📚 推荐阅读

- Hyvärinen & Oja (2000): ICA: Algorithms and Applications
- MNE ICA Tutorial: https://mne.tools/stable/auto_tutorials/preprocessing/40_artifact_correction_ica.html
- Makeig et al. (1996): ICA of Electroencephalographic Data
- Winkler et al. (2015): Automatic Classification of Artifactual ICA-Components

---

**🎉 恭喜完成全部 6 个单元！**
